# Experiment: J1-J2 Results Dashboard

Objective:
- Compare exact 4x4 references against the current baseline and champion recipes.
- Visualize what the model should reach versus what it currently reaches.
- Optionally launch a longer post-training evaluation of the champion from the notebook.


## What this notebook is for

This notebook gives a compact view of three things:

1. the fixed-panel search trajectory,
2. the current post-training benchmark against exact diagonalization,
3. an optional path to rerun the champion for longer budgets and redraw the same plots.


In [ ]:
from __future__ import annotations

import ast
import json
import subprocess
import sys
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs matplotlib. Install it in the kernel with `%pip install matplotlib`."
    ) from exc


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "prepare.py").exists() and (candidate / "train.py").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


ROOT = find_project_root(Path.cwd())
RESULTS_DIR = ROOT / "results"
ANALYSIS_DIR = ROOT / "analysis"

plt.style.use("seaborn-v0_8-whitegrid")
ROOT


In [ ]:
KEY_KNOBS = [
    "BATCH_SIZE",
    "LEARNING_RATE",
    "WEIGHT_DECAY",
    "GRAD_CLIP_NORM",
    "LR_WARMUP_FRACTION",
    "LR_FINAL_MULTIPLIER",
    "BASELINE_TYPE",
    "ADVANTAGE_TYPE",
]


def load_json(path: Path):
    return json.loads(path.read_text()) if path.exists() else None


def choose_latest_matching_path(base: Path, predicate) -> Path:
    matches = [path for path in base.iterdir() if path.is_dir() and predicate(path)] if base.exists() else []
    if not matches:
        raise RuntimeError(f"No matching run directory found under {base}")
    matches.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    return matches[0]


def load_ed_table() -> tuple[dict[float, float], dict[float, float]]:
    rows = json.loads((ANALYSIS_DIR / "ed_reference_table.json").read_text())
    sz0 = {}
    sz1 = {}
    for row in rows:
        j2 = float(row["j2"])
        if row["sector"] == "sz0":
            sz0[j2] = float(row["E0_per_site"])
        elif row["sector"] == "sz1":
            sz1[j2] = float(row["E0_per_site"])
    return sz0, sz1


def extract_literal_constants(path: Path) -> dict[str, object]:
    module = ast.parse(path.read_text())
    values: dict[str, object] = {}
    for node in module.body:
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        target = node.targets[0]
        if not isinstance(target, ast.Name) or not target.id.isupper():
            continue
        try:
            values[target.id] = ast.literal_eval(node.value)
        except Exception:
            continue
    return values


EXACT_SZ0, EXACT_SZ1 = load_ed_table()

DIRECT_SEARCH_DIR = choose_latest_matching_path(
    RESULTS_DIR,
    lambda path: (path / "best_train.py").exists() and (path / "ledger.jsonl").exists(),
)

ANCHOR_EVAL_DIR = choose_latest_matching_path(
    RESULTS_DIR,
    lambda path: (path / "summary.json").exists() and (path / "runs.jsonl").exists(),
)

DIRECT_SEARCH_DIR, ANCHOR_EVAL_DIR


## Search summary and recipe deltas

The next cell loads the current direct-edit search snapshots and shows which knobs changed between the initial and winning recipes.


In [ ]:
baseline_recipe = DIRECT_SEARCH_DIR / "initial_train.py"
champion_recipe = DIRECT_SEARCH_DIR / "best_train.py"
state = None
for candidate_name in ["experiment_state.json", "campaign_state.json", "state.json"]:
    candidate = DIRECT_SEARCH_DIR / candidate_name
    if candidate.exists():
        state = load_json(candidate)
        break
ledger_path = DIRECT_SEARCH_DIR / "ledger.jsonl"
ledger = [json.loads(line) for line in ledger_path.read_text().splitlines() if line.strip()] if ledger_path.exists() else []

baseline_constants = extract_literal_constants(baseline_recipe) if baseline_recipe.exists() else {}
champion_constants = extract_literal_constants(champion_recipe) if champion_recipe.exists() else {}

changed_knobs = {
    name: (baseline_constants.get(name), champion_constants.get(name))
    for name in KEY_KNOBS
    if baseline_constants.get(name) != champion_constants.get(name)
}

print(f"Direct-search directory: {DIRECT_SEARCH_DIR}")
print(f"Completed iterations: {state.get('completed_iterations') if state else 'n/a'}")
print(f"Best score: {state.get('best_score') if state else 'n/a'}")
print(f"Best iteration: {state.get('best_iteration') if state else 'n/a'}")
print("\nChanged key knobs:")
for name, (before, after) in changed_knobs.items():
    print(f"- {name}: {before!r} -> {after!r}")


In [ ]:
if ledger:
    iterations = [entry["iteration"] for entry in ledger]
    scores = [entry["panel_score"] for entry in ledger]
    statuses = [entry["status"] for entry in ledger]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(iterations, scores, marker="o", linewidth=2, label="panel score")
    kept_x = [it for it, status in zip(iterations, statuses) if status == "keep"]
    kept_y = [score for score, status in zip(scores, statuses) if status == "keep"]
    if kept_x:
        ax.scatter(kept_x, kept_y, s=60, color="tab:green", label="accepted")
    ax.set_title("Fixed-panel search trajectory")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Panel score (lower is better)")
    ax.legend()
    plt.show()
else:
    print("No ledger found for the selected direct-search directory.")


## Exact reference versus model output

The next cells answer the practical question: what should the system reach, and what did the baseline and champion actually reach on the current benchmark?


In [ ]:
summary = load_json(ANCHOR_EVAL_DIR / "summary.json")
if summary is None:
    raise RuntimeError(f"Missing summary.json in {ANCHOR_EVAL_DIR}")

print(f"Loaded post-training benchmark from: {ANCHOR_EVAL_DIR}")
for row in summary["overall"]:
    print(
        f"- {row['config_label']}: steps={row['max_steps']}, "
        f"MAE(E/N)={row['mae_energy_per_site_sz0']:.6f}, "
        f"MAE(gap)={row['mae_gap']:.6f}, "
        f"Var(E_loc)/N^2={row['mean_local_energy_var_per_site2_sz0']:.6f}"
    )


In [ ]:
per_j2 = summary["per_j2"]
labels = sorted({row['config_label'] for row in per_j2})
j2_values = sorted({row['j2'] for row in per_j2})
exact_by_j2 = {}
for row in per_j2:
    exact_by_j2.setdefault(row['j2'], row['exact_energy_per_site_sz0'])
exact_curve = [exact_by_j2[j2] if j2 in exact_by_j2 else EXACT_SZ0[j2] for j2 in j2_values]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].plot(j2_values, exact_curve, marker="o", linewidth=2.5, color="black", label="exact ED")
for label in labels:
    rows = sorted([row for row in per_j2 if row['config_label'] == label], key=lambda row: row['j2'])
    axes[0].plot(
        [row['j2'] for row in rows],
        [row['mean_energy_per_site_sz0'] for row in rows],
        marker="o",
        linewidth=2,
        label=label,
    )
axes[0].set_title("Energy per site: exact vs model")
axes[0].set_xlabel("J2/J1")
axes[0].set_ylabel("E/N")
axes[0].legend()

for label in labels:
    rows = sorted([row for row in per_j2 if row['config_label'] == label], key=lambda row: row['j2'])
    axes[1].plot(
        [row['j2'] for row in rows],
        [row['mae_energy_per_site_sz0'] for row in rows],
        marker="o",
        linewidth=2,
        label=label,
    )
axes[1].set_title("Energy error versus exact diagonalization")
axes[1].set_xlabel("J2/J1")
axes[1].set_ylabel("|E_model - E_exact| / N")
axes[1].legend()

plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

for label in labels:
    rows = sorted([row for row in per_j2 if row['config_label'] == label], key=lambda row: row['j2'])
    model_stripe = [max(row['mean_m_stripe_x_sq'], row['mean_m_stripe_y_sq']) for row in rows]
    exact_stripe = [max(row['exact_m_stripe_x_sq'], row['exact_m_stripe_y_sq']) for row in rows]
    axes[0].plot([row['j2'] for row in rows], [row['mean_local_energy_var_per_site2_sz0'] for row in rows], marker="o", linewidth=2, label=label)
    axes[1].plot([row['j2'] for row in rows], [row['mean_m_neel_sq'] for row in rows], marker="o", linewidth=2, label=f"{label} Neel")
    axes[1].plot([row['j2'] for row in rows], model_stripe, marker="s", linewidth=2, linestyle="--", label=f"{label} stripe")

axes[1].plot(j2_values, [summary_row['exact_m_neel_sq'] for summary_row in sorted([row for row in per_j2 if row['config_label'] == labels[0]], key=lambda row: row['j2'])], color="black", linewidth=2.5, alpha=0.7, label="exact Neel")
axes[1].plot(j2_values, [max(row['exact_m_stripe_x_sq'], row['exact_m_stripe_y_sq']) for row in sorted([row for row in per_j2 if row['config_label'] == labels[0]], key=lambda row: row['j2'])], color="gray", linewidth=2.5, alpha=0.7, linestyle=":", label="exact stripe")

axes[0].set_title("Variational quality")
axes[0].set_xlabel("J2/J1")
axes[0].set_ylabel("Var(E_loc) / N^2")
axes[0].legend()

axes[1].set_title("Order-parameter anchors")
axes[1].set_xlabel("J2/J1")
axes[1].set_ylabel("order parameter")
axes[1].legend(ncol=2)

plt.show()


## Optional: run a longer evaluation from here

Set `RUN_LONG_EVAL = True` in the next cell if you want to benchmark the current champion for longer training budgets and multiple seeds without leaving the notebook.


In [ ]:
BASELINE_PATH = baseline_recipe
CANDIDATE_PATH = champion_recipe
LONG_EVAL_DIR = RESULTS_DIR / "post_training_eval_longer"
J2_VALUES = [0.0, 0.4, 0.5, 0.6, 1.0]
SEEDS = [11, 22, 33]
STEPS = [50, 200]
METRIC_SAMPLES = 512
METRIC_REPEATS = 1
RUN_LONG_EVAL = False

print(f"baseline:  {BASELINE_PATH}")
print(f"candidate: {CANDIDATE_PATH}")
print(f"output:    {LONG_EVAL_DIR}")


In [ ]:
if RUN_LONG_EVAL:
    cmd = [
        sys.executable,
        str(ROOT / "controller" / "post_training_eval.py"),
        "--baseline-path", str(BASELINE_PATH),
        "--candidate-path", str(CANDIDATE_PATH),
        "--output-dir", str(LONG_EVAL_DIR),
        "--metric-samples", str(METRIC_SAMPLES),
        "--metric-repeats", str(METRIC_REPEATS),
        "--j2-values", *[str(x) for x in J2_VALUES],
        "--seeds", *[str(x) for x in SEEDS],
        "--steps", *[str(x) for x in STEPS],
    ]
    print("Running:")
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("Set RUN_LONG_EVAL = True and rerun this cell to launch the longer benchmark.")


## Suggested reading of the plots

- If the champion curve moves closer to the exact ED line at most `J2` anchors after a longer run, the search winner is turning into a genuinely better recipe.
- If the champion only wins on the fixed panel but not on the longer benchmark, then the search is improving short-budget optimization rather than final accuracy.
- The variance plot is useful even when energies look close: a lower `Var(E_loc)` usually means a better variational state.
